In [1]:

import sys
import subprocess
import json
import sqlite3
import textwrap
from pathlib import Path
from datetime import datetime, timezone
from xml.sax.saxutils import escape

try:
    import pandas as pd
    import numpy as np

    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "pandas", "numpy", "reportlab", "pyarrow"
    ])
    import pandas as pd
    import numpy as np

    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )

# ============================================================
# 1) PROJECT ROOT DISCOVERY
# ============================================================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
PREPARED_ROOT = PROJECT_ROOT / "data" / "prepared"
TRANSFORMED_ROOT = PROJECT_ROOT / "data" / "transformed"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
SILVER_ROOT = PROJECT_ROOT / "data" / "silver"
FEATURES_ROOT = PROJECT_ROOT / "data" / "features"

REPORTS_DIR = PROJECT_ROOT / "reports"
VALIDATION_DIR = REPORTS_DIR / "validation"
LOGS_DIR = PROJECT_ROOT / "logs"
SRC_DIR = PROJECT_ROOT / "src"
WAREHOUSE_DIR = PROJECT_ROOT / "warehouse"

FEATURE_STORE_DIR = PROJECT_ROOT / "feature_store"
FEATURE_STORE_DIR.mkdir(parents=True, exist_ok=True)

OFFLINE_STORE_DIR = FEATURE_STORE_DIR / "offline_store"
OFFLINE_STORE_DIR.mkdir(parents=True, exist_ok=True)

REGISTRY_DIR = FEATURE_STORE_DIR / "registry"
REGISTRY_DIR.mkdir(parents=True, exist_ok=True)

DOCS_DIR = FEATURE_STORE_DIR / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
FEATURE_VERSION = "v1"

OUTPUT_FILE_NAME = "07 Feature Store- DM4ML-Group51.pdf"
OUTPUT_PATH = PROJECT_ROOT / OUTPUT_FILE_NAME

FEATURE_STORE_DB = OFFLINE_STORE_DIR / "feature_store.db"
FEATURE_STORE_CONFIG_JSON = REGISTRY_DIR / "feature_store_config.json"
FEATURE_METADATA_CSV = REGISTRY_DIR / "feature_metadata.csv"
FEATURE_METADATA_JSON = REGISTRY_DIR / "feature_metadata.json"
FEATURE_STORE_SCHEMA_SQL = REGISTRY_DIR / "feature_store_schema.sql"
FEATURE_RETRIEVAL_TRAINING_CSV = DOCS_DIR / "sample_training_feature_retrieval.csv"
FEATURE_RETRIEVAL_INFERENCE_CSV = DOCS_DIR / "sample_inference_feature_retrieval.csv"
FEATURE_STORE_LOG = LOGS_DIR / f"feature_store_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"FEATURE_STORE_DIR: {FEATURE_STORE_DIR}")
print(f"FEATURE_STORE_DB: {FEATURE_STORE_DB}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")

# ============================================================
# 2) GENERIC HELPERS
# ============================================================
def safe_str(x):
    try:
        return str(x)
    except Exception:
        return ""

def rel_path(path):
    try:
        return safe_str(path.relative_to(PROJECT_ROOT))
    except Exception:
        return safe_str(path)

def log_event(stage, status, message, extra=None):
    record = {
        "event_ts": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "status": status,
        "message": message,
        "extra": extra or {},
    }
    FEATURE_STORE_LOG.parent.mkdir(parents=True, exist_ok=True)
    with open(FEATURE_STORE_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def collect_files(base_dir, patterns):
    results = []
    if not base_dir.exists():
        return results
    for pattern in patterns:
        results.extend(base_dir.rglob(pattern))
    return sorted(set(p for p in results if p.is_file()))

def latest_file(base_dir, patterns):
    files = collect_files(base_dir, patterns)
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

def file_info(path):
    if not path or not Path(path).exists():
        return None
    path = Path(path)
    st = path.stat()
    return {
        "name": path.name,
        "relative_path": rel_path(path),
        "size_kb": round(st.st_size / 1024, 2),
        "modified": datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "suffix": path.suffix.lower(),
    }

def read_text_preview(path, max_lines=80, max_chars=12000):
    if not path or not Path(path).exists():
        return "File not found."
    try:
        text = Path(path).read_text(encoding="utf-8", errors="ignore")
        lines = text.splitlines()[:max_lines]
        return "\n".join(lines)[:max_chars]
    except Exception as e:
        return f"Could not preview file: {e}"

def read_json_preview(path, max_chars=12000):
    if not path or not Path(path).exists():
        return "JSON file not found."
    try:
        with open(path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        return json.dumps(payload, indent=2, ensure_ascii=False)[:max_chars]
    except Exception as e:
        return f"Could not read JSON preview: {e}"

def read_notebook_preview(path, max_code_cells=4, max_chars=9000):
    if not path or not Path(path).exists():
        return "Notebook not found."
    try:
        with open(path, "r", encoding="utf-8") as f:
            nb = json.load(f)
        parts = []
        code_idx = 0
        for cell in nb.get("cells", []):
            if cell.get("cell_type") != "code":
                continue
            code_idx += 1
            src = cell.get("source", [])
            src = "".join(src) if isinstance(src, list) else str(src)
            src = src.strip()
            if src:
                parts.append(f"# Code cell {code_idx}\n{src}")
            if len("\n\n".join(parts)) >= max_chars or code_idx >= max_code_cells:
                break
        out = "\n\n".join(parts).strip()
        return out[:max_chars] if out else "No code cells found."
    except Exception as e:
        return f"Could not parse notebook: {e}"

def read_code_preview(path, max_lines=140, max_chars=9000):
    if not path or not Path(path).exists():
        return "File not found."
    if Path(path).suffix.lower() == ".ipynb":
        return read_notebook_preview(path, max_code_cells=4, max_chars=max_chars)
    return read_text_preview(path, max_lines=max_lines, max_chars=max_chars)

def build_tree_text(base_path, max_depth=5, max_items=200):
    base_path = Path(base_path)
    if not base_path.exists():
        return f"{base_path.name}/ (not found)"
    lines = [f"{base_path.name}/"]
    count = 0

    def walk(path, prefix="", depth=0):
        nonlocal count
        if depth >= max_depth or count >= max_items:
            return
        items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        for idx, item in enumerate(items):
            if count >= max_items:
                break
            connector = "└── " if idx == len(items) - 1 else "├── "
            lines.append(prefix + connector + item.name + ("/" if item.is_dir() else ""))
            count += 1
            if item.is_dir():
                extension = "    " if idx == len(items) - 1 else "│   "
                walk(item, prefix + extension, depth + 1)

    walk(base_path)
    if count >= max_items:
        lines.append("... output truncated ...")
    return "\n".join(lines)

def wrap_block_text(text, width=95):
    wrapped = []
    for line in str(text).splitlines():
        if not line.strip():
            wrapped.append("")
            continue
        pieces = textwrap.wrap(
            line,
            width=width,
            break_long_words=True,
            break_on_hyphens=True,
            replace_whitespace=False,
            drop_whitespace=False,
        )
        wrapped.extend(pieces if pieces else [""])
    return "\n".join(wrapped)

def make_hashable_value(v):
    if isinstance(v, np.ndarray):
        return tuple(make_hashable_value(x) for x in v.tolist())
    if isinstance(v, list):
        return tuple(make_hashable_value(x) for x in v)
    if isinstance(v, tuple):
        return tuple(make_hashable_value(x) for x in v)
    if isinstance(v, set):
        return tuple(sorted(make_hashable_value(x) for x in v))
    if isinstance(v, dict):
        return json.dumps(v, sort_keys=True, ensure_ascii=False, default=str)
    try:
        hash(v)
        return v
    except TypeError:
        return str(v)

def make_display_value(v):
    if isinstance(v, np.ndarray):
        return json.dumps(v.tolist(), ensure_ascii=False)
    if isinstance(v, (list, tuple, set)):
        try:
            return json.dumps(list(v), ensure_ascii=False)
        except Exception:
            return str(v)
    if isinstance(v, dict):
        try:
            return json.dumps(v, ensure_ascii=False, sort_keys=True, default=str)
        except Exception:
            return str(v)
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    return safe_str(v)

def safe_duplicate_count(df):
    if df is None or df.empty:
        return 0
    tmp = df.copy()
    for col in tmp.columns:
        tmp[col] = tmp[col].map(make_hashable_value)
    return int(tmp.duplicated().sum())

def wrap_path_for_pdf(value, max_chunk=32):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    separators = {"\\", "/", "_", "-", "="}
    parts = []
    token = ""

    for ch in text:
        token += ch
        if ch in separators:
            parts.append(token)
            token = ""
    if token:
        parts.append(token)

    lines = []
    current = ""

    for part in parts:
        if len(current) + len(part) <= max_chunk:
            current += part
        else:
            if current:
                lines.append(current)
            if len(part) <= max_chunk:
                current = part
            else:
                subparts = textwrap.wrap(
                    part,
                    width=max_chunk,
                    break_long_words=True,
                    break_on_hyphens=True,
                )
                if subparts:
                    lines.extend(subparts[:-1])
                    current = subparts[-1]
                else:
                    current = part

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def wrap_general_text_for_pdf(value, max_len=36):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    words = text.split()
    lines = []
    current = ""

    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_len:
            current = candidate
        else:
            if current:
                lines.append(current)
            if len(word) > max_len:
                chunks = textwrap.wrap(
                    word,
                    width=max_len,
                    break_long_words=True,
                    break_on_hyphens=True,
                )
                if chunks:
                    lines.extend(chunks[:-1])
                    current = chunks[-1]
                else:
                    current = word
            else:
                current = word

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def load_table_file(path, nrows=None):
    if not path or not Path(path).exists():
        return None
    path = Path(path)
    try:
        if path.suffix.lower() == ".csv":
            return pd.read_csv(path, nrows=nrows)
        if path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
            return df.head(nrows) if nrows else df
        if path.suffix.lower() == ".json":
            with open(path, "r", encoding="utf-8") as f:
                payload = json.load(f)
            if isinstance(payload, list):
                return pd.DataFrame(payload).head(nrows) if nrows else pd.DataFrame(payload)
            if isinstance(payload, dict):
                for v in payload.values():
                    if isinstance(v, list) and v and isinstance(v[0], dict):
                        df = pd.DataFrame(v)
                        return df.head(nrows) if nrows else df
                return pd.DataFrame([payload]).head(nrows) if nrows else pd.DataFrame([payload])
    except Exception as e:
        print(f"Could not load {path}: {e}")
        return None
    return None

def load_sqlite_table(db_path, table_name, nrows=None):
    if not db_path or not Path(db_path).exists():
        return None
    conn = sqlite3.connect(db_path)
    try:
        query = f"SELECT * FROM {table_name}"
        if nrows:
            query += f" LIMIT {int(nrows)}"
        df = pd.read_sql_query(query, conn)
        conn.close()
        return df
    except Exception:
        conn.close()
        return None

# ============================================================
# 3) DISCOVERY HELPERS
# ============================================================
def discover_events_file():
    patterns = [
        "**/events.csv",
        "**/*events*.csv",
        "**/*interaction*.csv",
        "**/*clickstream*.csv",
        "**/*transactions*.csv",
    ]
    matches = []
    for pattern in patterns:
        if RAW_ROOT.exists():
            matches.extend(RAW_ROOT.rglob(pattern))
    return max(matches, key=lambda p: p.stat().st_mtime) if matches else None

def discover_products_file():
    patterns = [
        "**/products.parquet",
        "**/products.csv",
        "**/products_raw.json",
        "**/*product*.parquet",
        "**/*product*.csv",
        "**/*product*.json",
    ]
    matches = []
    for base in [BRONZE_ROOT, RAW_ROOT]:
        if base.exists():
            for pattern in patterns:
                matches.extend(base.rglob(pattern))
    return max(matches, key=lambda p: p.stat().st_mtime) if matches else None

def discover_prepared_interactions_file():
    patterns = [
        "**/prepared_interactions*.parquet",
        "**/prepared_interactions*.csv",
        "**/*prepared*interaction*.parquet",
        "**/*prepared*interaction*.csv",
    ]
    matches = []
    for base in [PREPARED_ROOT, PROCESSED_ROOT, SILVER_ROOT]:
        if base.exists():
            for pattern in patterns:
                matches.extend(base.rglob(pattern))
    return max(matches, key=lambda p: p.stat().st_mtime) if matches else None

def discover_feature_table_file(name_root):
    patterns = [
        f"**/{name_root}.csv",
        f"**/{name_root}.parquet",
        f"**/{name_root}_*.csv",
        f"**/{name_root}_*.parquet",
    ]
    matches = []
    for base in [TRANSFORMED_ROOT, FEATURES_ROOT, PROCESSED_ROOT, SILVER_ROOT]:
        if base.exists():
            for pattern in patterns:
                matches.extend(base.rglob(pattern))
    return max(matches, key=lambda p: p.stat().st_mtime) if matches else None

def discover_feature_engineering_db():
    patterns = [
        "**/recommendation_features.db",
        "**/*feature*.db",
        "**/*warehouse*.db",
    ]
    matches = []
    for base in [WAREHOUSE_DIR, PROJECT_ROOT]:
        if base.exists():
            for pattern in patterns:
                matches.extend(base.rglob(pattern))
    return max(matches, key=lambda p: p.stat().st_mtime) if matches else None

def find_feature_store_assets():
    patterns = [
        "*feature*store*.py", "*feature*store*.ipynb", "*feature*store*.sql", "*feature*store*.json",
        "*feast*.py", "*feast*.ipynb", "*feast*.yaml", "*feast*.yml",
        "*registry*.py", "*registry*.ipynb", "*registry*.sql", "*registry*.json",
        "*feature*.py", "*feature*.ipynb", "*feature*.sql",
        "*transform*.py", "*transform*.ipynb", "*transform*.sql",
        "*schema*.sql", "*schema*.py",
    ]
    matches = []
    for base in [PROJECT_ROOT, SRC_DIR]:
        if base.exists():
            for pattern in patterns:
                matches.extend(base.rglob(pattern))

    cleaned = []
    for p in sorted(set(matches)):
        p_str = safe_str(p).lower()
        if ".ipynb_checkpoints" in p_str:
            continue
        if "/venv/" in p_str or "\\venv\\" in p_str or "/.venv/" in p_str or "\\.venv\\" in p_str:
            continue
        if "/site-packages/" in p_str or "\\site-packages\\" in p_str:
            continue
        cleaned.append(p)
    return cleaned

# ============================================================
# 4) FEATURE SOURCE PREPARATION
# ============================================================
def standardize_events(df):
    if df is None or df.empty:
        return None, []

    notes = []
    df = df.copy()

    rename_map = {}
    lower_map = {c.lower(): c for c in df.columns}

    if "visitorid" in lower_map:
        rename_map[lower_map["visitorid"]] = "user_id"
    elif "userid" in lower_map:
        rename_map[lower_map["userid"]] = "user_id"

    if "itemid" in lower_map:
        rename_map[lower_map["itemid"]] = "item_id"
    elif "productid" in lower_map:
        rename_map[lower_map["productid"]] = "item_id"

    if "timestamp" in lower_map:
        rename_map[lower_map["timestamp"]] = "event_ts"
    elif "event_ts" in lower_map:
        rename_map[lower_map["event_ts"]] = "event_ts"

    if "event" in lower_map:
        rename_map[lower_map["event"]] = "event_type"
    elif "eventtype" in lower_map:
        rename_map[lower_map["eventtype"]] = "event_type"

    if "rating" in lower_map:
        rename_map[lower_map["rating"]] = "rating"
    elif "score" in lower_map:
        rename_map[lower_map["score"]] = "rating"

    df = df.rename(columns=rename_map)
    notes.append("Standardized interaction column names where present.")

    needed = [c for c in ["user_id", "item_id"] if c in df.columns]
    if needed:
        before = len(df)
        df = df.dropna(subset=needed)
        notes.append(f"Dropped rows missing required interaction keys: {before - len(df)} removed.")

    if "event_type" in df.columns:
        df["event_type"] = df["event_type"].astype(str).str.strip().str.lower()
    else:
        df["event_type"] = "interaction"

    if "event_ts" in df.columns:
        ts_num = pd.to_numeric(df["event_ts"], errors="coerce")
        if ts_num.notna().sum() > 0:
            median_val = ts_num.dropna().median()
            if median_val > 1e12:
                df["event_ts"] = pd.to_datetime(ts_num, unit="ms", errors="coerce")
            elif median_val > 1e9:
                df["event_ts"] = pd.to_datetime(ts_num, unit="s", errors="coerce")
            else:
                df["event_ts"] = pd.to_datetime(df["event_ts"], errors="coerce")
        else:
            df["event_ts"] = pd.to_datetime(df["event_ts"], errors="coerce")
    else:
        df["event_ts"] = pd.NaT

    if "rating" in df.columns:
        df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

    weight_map = {
        "view": 1.0,
        "click": 1.0,
        "interaction": 1.0,
        "addtocart": 3.0,
        "cart": 3.0,
        "purchase": 5.0,
        "transaction": 5.0,
    }
    df["interaction_weight"] = df["event_type"].map(weight_map).fillna(1.0)

    dedupe_cols = [c for c in ["user_id", "item_id", "event_type", "event_ts"] if c in df.columns]
    if dedupe_cols:
        before = len(df)
        df = df.drop_duplicates(subset=dedupe_cols)
        notes.append(f"Removed duplicate interactions using {', '.join(dedupe_cols)}: {before - len(df)} removed.")

    if "user_id" in df.columns:
        df["user_id"] = df["user_id"].astype(str)
    if "item_id" in df.columns:
        df["item_id"] = df["item_id"].astype(str)

    return df, notes

def standardize_products(df):
    if df is None or df.empty:
        return None, []

    notes = []
    df = df.copy()

    for col in df.columns:
        df[col] = df[col].map(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)

    lower_map = {c.lower(): c for c in df.columns}
    rename_map = {}

    if "id" in lower_map and "product_id" not in df.columns:
        rename_map[lower_map["id"]] = "product_id"
    if "title" in lower_map:
        rename_map[lower_map["title"]] = "title"
    if "category" in lower_map:
        rename_map[lower_map["category"]] = "category"
    if "price" in lower_map:
        rename_map[lower_map["price"]] = "price"

    df = df.rename(columns=rename_map)
    notes.append("Standardized product column names where present.")

    if "product_id" in df.columns:
        df["product_id"] = df["product_id"].astype(str)

    if "category" in df.columns:
        df["category"] = df["category"].astype("string").fillna("unknown").str.strip().str.lower()

    if "price" in df.columns:
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
        median_price = df["price"].median()
        if pd.notna(median_price):
            df["price"] = df["price"].fillna(median_price)
        pmin, pmax = df["price"].min(), df["price"].max()
        if pd.notna(pmin) and pd.notna(pmax) and pmax != pmin:
            df["price_norm"] = (df["price"] - pmin) / (pmax - pmin)
        else:
            df["price_norm"] = 0.0
        notes.append("Prepared product price fields for downstream feature joins.")

    if "product_id" in df.columns:
        before = len(df)
        pid = df["product_id"].map(make_hashable_value)
        df = df.loc[~pid.duplicated()].copy()
        notes.append(f"Removed duplicate product_id rows: {before - len(df)} removed.")

    return df, notes

def build_feature_tables_from_base(events_df, products_df):
    outputs = {
        "user_features": pd.DataFrame(),
        "item_features": pd.DataFrame(),
        "user_item_features": pd.DataFrame(),
        "item_cooccurrence": pd.DataFrame(),
        "feature_logic": [],
    }

    if events_df is None or events_df.empty:
        outputs["feature_logic"].append("No interaction dataset was available, so feature tables could not be created.")
        return outputs

    if not {"user_id", "item_id"}.issubset(set(events_df.columns)):
        outputs["feature_logic"].append("Interaction data did not contain user_id and item_id, so feature tables could not be created.")
        return outputs

    work = events_df.copy()

    user_features = (
        work.groupby("user_id", dropna=False)
        .agg(
            activity_frequency=("item_id", "count"),
            unique_items_interacted=("item_id", pd.Series.nunique),
            avg_interaction_weight=("interaction_weight", "mean"),
        )
        .reset_index()
    )

    if "rating" in work.columns and work["rating"].notna().sum() > 0:
        user_rating = work.groupby("user_id", dropna=False)["rating"].mean().reset_index(name="avg_rating_per_user")
        item_rating = work.groupby("item_id", dropna=False)["rating"].mean().reset_index(name="avg_rating_per_item")
        rating_logic = "Average rating features were computed from the rating column."
    else:
        user_rating = work.groupby("user_id", dropna=False)["interaction_weight"].mean().reset_index(name="avg_rating_per_user")
        item_rating = work.groupby("item_id", dropna=False)["interaction_weight"].mean().reset_index(name="avg_rating_per_item")
        rating_logic = "No explicit rating column was available, so weighted interaction score was used as a proxy average rating."

    user_features = user_features.merge(user_rating, on="user_id", how="left")

    item_features = (
        work.groupby("item_id", dropna=False)
        .agg(
            item_interaction_count=("user_id", "count"),
            unique_users=("user_id", pd.Series.nunique),
            avg_interaction_weight=("interaction_weight", "mean"),
        )
        .reset_index()
        .merge(item_rating, on="item_id", how="left")
    )

    if products_df is not None and not products_df.empty and "product_id" in products_df.columns:
        join_cols = ["product_id"]
        for c in ["title", "category", "price", "price_norm"]:
            if c in products_df.columns:
                join_cols.append(c)
        item_features = item_features.merge(
            products_df[join_cols],
            left_on="item_id",
            right_on="product_id",
            how="left"
        )
        if "product_id" in item_features.columns:
            item_features = item_features.drop(columns=["product_id"])

    user_item_features = (
        work.groupby(["user_id", "item_id"], dropna=False)
        .agg(
            interaction_count=("event_type", "count"),
            total_interaction_weight=("interaction_weight", "sum"),
            last_event_ts=("event_ts", "max"),
        )
        .reset_index()
    )

    event_counts = pd.crosstab([work["user_id"], work["item_id"]], work["event_type"]).reset_index()
    user_item_features = user_item_features.merge(event_counts, on=["user_id", "item_id"], how="left")

    pairs = work[["user_id", "item_id"]].drop_duplicates()
    pair_rows = []
    grouped = pairs.groupby("user_id")["item_id"].apply(list)

    for _, item_list in grouped.items():
        item_list = [str(x) for x in item_list if pd.notna(x)]
        item_list = sorted(set(item_list))
        for i in range(len(item_list)):
            for j in range(i + 1, len(item_list)):
                pair_rows.append((item_list[i], item_list[j], 1))

    if pair_rows:
        item_cooccurrence = pd.DataFrame(pair_rows, columns=["item_id_a", "item_id_b", "cooccurrence_count"])
        item_cooccurrence = (
            item_cooccurrence.groupby(["item_id_a", "item_id_b"], as_index=False)["cooccurrence_count"]
            .sum()
            .sort_values(["cooccurrence_count", "item_id_a", "item_id_b"], ascending=[False, True, True])
        )
    else:
        item_cooccurrence = pd.DataFrame(columns=["item_id_a", "item_id_b", "cooccurrence_count"])

    outputs["user_features"] = user_features
    outputs["item_features"] = item_features
    outputs["user_item_features"] = user_item_features
    outputs["item_cooccurrence"] = item_cooccurrence
    outputs["feature_logic"] = [
        "User activity frequency = count of interactions per user.",
        rating_logic,
        "Item popularity = interaction count and distinct users per item.",
        "User-item features = interaction count, total interaction weight, latest event timestamp, and event-type counts.",
        "Co-occurrence features = item pair counts from shared user histories.",
    ]
    return outputs

def save_dataframe(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)

# ============================================================
# 5) FEATURE STORE BUILDERS
# ============================================================
def ensure_feature_sources():
    log_event("feature_source", "started", "Discovering feature source tables")

    paths = {
        "user_features": discover_feature_table_file("user_features"),
        "item_features": discover_feature_table_file("item_features"),
        "user_item_features": discover_feature_table_file("user_item_features"),
        "item_cooccurrence": discover_feature_table_file("item_cooccurrence"),
    }

    db_candidate = discover_feature_engineering_db()

    tables = {}
    for name, path in paths.items():
        df = load_table_file(path) if path else None
        if (df is None or df.empty) and db_candidate:
            df = load_sqlite_table(db_candidate, name)
        tables[name] = df if df is not None else pd.DataFrame()

    if all(df is not None and not df.empty for df in tables.values()):
        log_event("feature_source", "completed", "Loaded existing transformed feature tables", {
            "db_candidate": rel_path(db_candidate) if db_candidate else None,
            "paths": {k: rel_path(v) if v else None for k, v in paths.items()}
        })
        return tables, paths, db_candidate, ["Loaded feature tables from existing transformed data or warehouse database."]

    events_path = discover_prepared_interactions_file()
    if not events_path:
        events_path = discover_events_file()
    products_path = discover_products_file()

    events_df = load_table_file(events_path)
    products_df = load_table_file(products_path)

    events_df, event_notes = standardize_events(events_df)
    products_df, product_notes = standardize_products(products_df)

    rebuilt = build_feature_tables_from_base(events_df, products_df)

    path_map = {}
    for name in ["user_features", "item_features", "user_item_features", "item_cooccurrence"]:
        df = rebuilt.get(name, pd.DataFrame())
        if df is not None and not df.empty:
            out_path = FEATURES_ROOT / f"{name}.csv"
            save_dataframe(df, out_path)
            path_map[name] = out_path
            tables[name] = df
        else:
            path_map[name] = None
            tables[name] = pd.DataFrame()

    notes = []
    notes.extend(event_notes if event_notes else [])
    notes.extend(product_notes if product_notes else [])
    notes.extend(rebuilt.get("feature_logic", []))
    notes.append("Feature tables were rebuilt because complete transformed sources were not already available.")
    log_event("feature_source", "completed", "Feature tables prepared for feature store", {
        "events_source": rel_path(events_path) if events_path else None,
        "products_source": rel_path(products_path) if products_path else None,
    })
    return tables, path_map, db_candidate, notes

def feature_transformation_description(entity_name, feature_name):
    lookup = {
        "activity_frequency": "Count of interactions per user.",
        "unique_items_interacted": "Distinct item count per user.",
        "avg_interaction_weight": "Mean interaction weight based on event type mapping.",
        "avg_rating_per_user": "Average explicit rating or proxy weighted score per user.",
        "item_interaction_count": "Count of interactions per item.",
        "unique_users": "Distinct user count per item.",
        "avg_rating_per_item": "Average explicit rating or proxy weighted score per item.",
        "interaction_count": "Count of user-item interactions.",
        "total_interaction_weight": "Sum of interaction weights for the user-item pair.",
        "last_event_ts": "Most recent interaction timestamp for the user-item pair.",
        "cooccurrence_count": "Count of times two items appeared in the same user history.",
        "view": "Count of view events for a user-item pair.",
        "click": "Count of click events for a user-item pair.",
        "addtocart": "Count of add-to-cart events for a user-item pair.",
        "cart": "Count of cart events for a user-item pair.",
        "purchase": "Count of purchase events for a user-item pair.",
        "transaction": "Count of transaction events for a user-item pair.",
        "title": "Joined item title from product metadata.",
        "category": "Joined item category from product metadata.",
        "price": "Joined product price.",
        "price_norm": "Normalized product price.",
    }
    return lookup.get(feature_name, f"Derived {feature_name} feature for {entity_name}.")

def build_feature_metadata(feature_tables, source_path_map):
    rows = []
    entity_keys = {
        "user_features": ["user_id"],
        "item_features": ["item_id"],
        "user_item_features": ["user_id", "item_id"],
        "item_cooccurrence": ["item_id_a", "item_id_b"],
    }

    for table_name, df in feature_tables.items():
        if df is None or df.empty:
            continue
        source_path = source_path_map.get(table_name)
        keys = entity_keys.get(table_name, [])
        for col in df.columns:
            if col in keys:
                continue
            rows.append({
                "feature_view": table_name,
                "entity_keys": ", ".join(keys),
                "feature_name": col,
                "dtype": safe_str(df[col].dtype),
                "source_path": rel_path(source_path) if source_path else "generated_in_notebook",
                "source_table": table_name,
                "transformation_logic": feature_transformation_description(table_name, col),
                "version": FEATURE_VERSION,
                "retrieval_modes": "training,inference",
                "created_ts": RUN_TS,
            })
    meta_df = pd.DataFrame(rows)
    return meta_df.sort_values(["feature_view", "feature_name"]).reset_index(drop=True) if not meta_df.empty else meta_df

def build_feature_store_config(feature_tables, source_path_map):
    feature_views = []
    for name, df in feature_tables.items():
        if df is None or df.empty:
            continue
        entities = {
            "user_features": ["user_id"],
            "item_features": ["item_id"],
            "user_item_features": ["user_id", "item_id"],
            "item_cooccurrence": ["item_id_a", "item_id_b"],
        }.get(name, [])
        feature_cols = [c for c in df.columns if c not in entities]
        feature_views.append({
            "name": name,
            "version": FEATURE_VERSION,
            "entities": entities,
            "source_path": rel_path(source_path_map.get(name)) if source_path_map.get(name) else "generated_in_notebook",
            "feature_count": len(feature_cols),
            "features": feature_cols,
            "online_enabled": True,
            "offline_enabled": True,
        })

    cfg = {
        "project": "RecoMart Feature Store",
        "store_type": "custom_metadata_registry",
        "version": FEATURE_VERSION,
        "created_ts": RUN_TS,
        "entities": ["user_id", "item_id"],
        "offline_store": {
            "type": "sqlite",
            "path": rel_path(FEATURE_STORE_DB),
        },
        "registry": {
            "metadata_csv": rel_path(FEATURE_METADATA_CSV),
            "metadata_json": rel_path(FEATURE_METADATA_JSON),
        },
        "feature_views": feature_views,
    }
    return cfg

def write_feature_store(feature_tables, metadata_df, config_dict):
    log_event("feature_store", "started", "Writing feature store artifacts")
    conn = sqlite3.connect(FEATURE_STORE_DB)

    version_df = pd.DataFrame([{
        "version": FEATURE_VERSION,
        "created_ts": RUN_TS,
        "store_type": "custom_metadata_registry",
        "offline_store": rel_path(FEATURE_STORE_DB),
        "config_path": rel_path(FEATURE_STORE_CONFIG_JSON),
        "metadata_path": rel_path(FEATURE_METADATA_CSV),
    }])
    version_df.to_sql("feature_store_versions", conn, if_exists="replace", index=False)

    if metadata_df is not None and not metadata_df.empty:
        metadata_df.to_sql("feature_metadata", conn, if_exists="replace", index=False)

    for table_name, df in feature_tables.items():
        if df is not None and not df.empty:
            df.to_sql(f"{table_name}_{FEATURE_VERSION}", conn, if_exists="replace", index=False)

    conn.commit()
    conn.close()

    FEATURE_STORE_CONFIG_JSON.write_text(json.dumps(config_dict, indent=2, ensure_ascii=False), encoding="utf-8")
    metadata_df.to_csv(FEATURE_METADATA_CSV, index=False)
    FEATURE_METADATA_JSON.write_text(metadata_df.to_json(orient="records", indent=2, force_ascii=False), encoding="utf-8")

    log_event("feature_store", "completed", "Feature store artifacts written", {
        "db": rel_path(FEATURE_STORE_DB),
        "config_json": rel_path(FEATURE_STORE_CONFIG_JSON),
        "metadata_csv": rel_path(FEATURE_METADATA_CSV),
        "metadata_json": rel_path(FEATURE_METADATA_JSON),
    })

def export_feature_store_schema():
    conn = sqlite3.connect(FEATURE_STORE_DB)
    cur = conn.cursor()
    cur.execute("""
        SELECT sql
        FROM sqlite_master
        WHERE type IN ('table', 'index')
          AND name NOT LIKE 'sqlite_%'
        ORDER BY type, name
    """)
    statements = [row[0] for row in cur.fetchall() if row[0]]
    conn.close()

    schema_text = ";\n\n".join(statements).strip()
    if schema_text:
        schema_text += ";"
    else:
        schema_text = "-- No schema objects found."
    FEATURE_STORE_SCHEMA_SQL.write_text(schema_text, encoding="utf-8")
    return schema_text

def sample_training_feature_retrieval(limit=10):
    conn = sqlite3.connect(FEATURE_STORE_DB)
    try:
        query = f"""
            SELECT
                ui.user_id,
                ui.item_id,
                ui.interaction_count,
                ui.total_interaction_weight,
                ui.last_event_ts,
                uf.activity_frequency,
                uf.unique_items_interacted,
                uf.avg_rating_per_user,
                it.item_interaction_count,
                it.unique_users,
                it.avg_rating_per_item,
                it.category,
                it.price_norm
            FROM user_item_features_{FEATURE_VERSION} ui
            LEFT JOIN user_features_{FEATURE_VERSION} uf
                ON ui.user_id = uf.user_id
            LEFT JOIN item_features_{FEATURE_VERSION} it
                ON ui.item_id = it.item_id
            LIMIT {int(limit)}
        """
        df = pd.read_sql_query(query, conn)
    except Exception:
        df = pd.DataFrame()
    conn.close()
    return df

def sample_inference_feature_retrieval(limit=10):
    conn = sqlite3.connect(FEATURE_STORE_DB)
    try:
        ui = pd.read_sql_query(f"SELECT * FROM user_item_features_{FEATURE_VERSION} LIMIT {int(limit)}", conn)
        if ui.empty:
            conn.close()
            return pd.DataFrame()

        sample_user = safe_str(ui.iloc[0]["user_id"])
        sample_items = ui["item_id"].astype(str).head(limit).tolist()

        placeholders = ",".join(["?"] * len(sample_items))
        query = f"""
            SELECT
                ? AS request_user_id,
                it.item_id,
                it.item_interaction_count,
                it.unique_users,
                it.avg_rating_per_item,
                it.category,
                it.price_norm
            FROM item_features_{FEATURE_VERSION} it
            WHERE it.item_id IN ({placeholders})
        """
        params = [sample_user] + sample_items
        df = pd.read_sql_query(query, conn, params=params)
    except Exception:
        df = pd.DataFrame()
    conn.close()
    return df

# ============================================================
# 6) PDF STYLES AND TABLE HELPERS
# ============================================================
styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    name="CustomTitle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    fontSize=16,
    leading=20,
    spaceAfter=14,
)

meta_style = ParagraphStyle(
    name="MetaStyle",
    parent=styles["Normal"],
    alignment=TA_LEFT,
    fontSize=10.2,
    leading=13,
    spaceAfter=5,
)

heading_style = ParagraphStyle(
    name="HeadingStyle",
    parent=styles["Heading2"],
    alignment=TA_LEFT,
    fontSize=12,
    leading=15,
    spaceAfter=8,
)

sub_heading_style = ParagraphStyle(
    name="SubHeadingStyle",
    parent=styles["Heading3"],
    alignment=TA_LEFT,
    fontSize=10.2,
    leading=12.5,
    spaceAfter=6,
)

body_style = ParagraphStyle(
    name="BodyStyle",
    parent=styles["BodyText"],
    alignment=TA_JUSTIFY,
    fontSize=10.0,
    leading=14,
    spaceAfter=8,
)

bullet_style = ParagraphStyle(
    name="BulletStyle",
    parent=styles["BodyText"],
    alignment=TA_LEFT,
    fontSize=10.0,
    leading=14,
    leftIndent=14,
    firstLineIndent=-8,
    spaceAfter=4,
)

code_style = ParagraphStyle(
    name="CodeStyle",
    parent=styles["Code"],
    fontName="Courier",
    fontSize=7.0,
    leading=8.4,
)

table_header_style = ParagraphStyle(
    name="TableHeaderStyle",
    parent=styles["BodyText"],
    fontName="Helvetica-Bold",
    fontSize=8.1,
    leading=9.4,
    alignment=TA_LEFT,
)

table_cell_style = ParagraphStyle(
    name="TableCellStyle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=7.0,
    leading=8.5,
    alignment=TA_LEFT,
)

def to_para(value, style, kind="general"):
    if kind == "path":
        return Paragraph(wrap_path_for_pdf(value), style)
    return Paragraph(wrap_general_text_for_pdf(value), style)

def make_wrapped_table(data, col_widths=None, header_bg="#D9EAD3", path_cols=None, file_cols=None):
    path_cols = path_cols or []
    file_cols = file_cols or []
    converted = []

    for r, row in enumerate(data):
        row_cells = []
        for c, cell in enumerate(row):
            style = table_header_style if r == 0 else table_cell_style
            if r == 0:
                row_cells.append(Paragraph(escape(str(cell)), style))
            else:
                if c in path_cols or c in file_cols:
                    row_cells.append(to_para(cell, style, kind="path"))
                else:
                    row_cells.append(to_para(cell, style, kind="general"))
        converted.append(row_cells)

    table = Table(converted, colWidths=col_widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(header_bg)),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    return table

def df_to_wrapped_table(df, max_rows=20, col_widths=None):
    if df is None or df.empty:
        data = [["No data available"]]
    else:
        preview = df.head(max_rows).copy()
        for col in preview.columns:
            preview[col] = preview[col].map(make_display_value)
        data = [list(preview.columns)] + preview.astype(str).values.tolist()
    return make_wrapped_table(data, col_widths=col_widths)

# ============================================================
# 7) BUILD FEATURE STORE ARTIFACTS
# ============================================================
log_event("pipeline", "started", "Feature store notebook execution started")

feature_tables, feature_source_paths, existing_feature_db, source_notes = ensure_feature_sources()
feature_metadata_df = build_feature_metadata(feature_tables, feature_source_paths)
feature_store_config = build_feature_store_config(feature_tables, feature_source_paths)
write_feature_store(feature_tables, feature_metadata_df, feature_store_config)
feature_store_schema_text = export_feature_store_schema()

training_retrieval_df = sample_training_feature_retrieval(limit=10)
inference_retrieval_df = sample_inference_feature_retrieval(limit=10)

if training_retrieval_df is not None and not training_retrieval_df.empty:
    training_retrieval_df.to_csv(FEATURE_RETRIEVAL_TRAINING_CSV, index=False)
if inference_retrieval_df is not None and not inference_retrieval_df.empty:
    inference_retrieval_df.to_csv(FEATURE_RETRIEVAL_INFERENCE_CSV, index=False)

log_event("pipeline", "completed", "Feature store notebook execution completed", {
    "output_pdf": rel_path(OUTPUT_PATH),
    "feature_store_db": rel_path(FEATURE_STORE_DB),
    "feature_metadata_csv": rel_path(FEATURE_METADATA_CSV),
})

# ============================================================
# 8) COLLECT PROJECT / LOG EVIDENCE
# ============================================================
feature_store_assets = find_feature_store_assets()

latest_validation_txt = latest_file(PROJECT_ROOT, ["**/run_validation.txt"])
latest_validation_json = latest_file(VALIDATION_DIR, ["**/data_quality_report_*.json"])
latest_validation_pdf = latest_file(VALIDATION_DIR, ["**/data_quality_report_*.pdf"])
latest_fix_log = latest_file(VALIDATION_DIR, ["**/fix_log_*.csv"])
latest_reval_summary = latest_file(VALIDATION_DIR, ["**/validation_summary_revalidated_*.csv"])
latest_reval_issues = latest_file(VALIDATION_DIR, ["**/validation_issues_revalidated_*.csv"])
latest_validation_log = latest_file(LOGS_DIR, ["**/validation_log_*.jsonl"])

events_file = discover_events_file()
products_file = discover_products_file()
prepared_interactions_file = discover_prepared_interactions_file()

source_table_summary_df = pd.DataFrame([
    {
        "table_name": "user_features",
        "rows": 0 if feature_tables["user_features"].empty else len(feature_tables["user_features"]),
        "columns": 0 if feature_tables["user_features"].empty else len(feature_tables["user_features"].columns),
        "duplicate_rows": safe_duplicate_count(feature_tables["user_features"]),
    },
    {
        "table_name": "item_features",
        "rows": 0 if feature_tables["item_features"].empty else len(feature_tables["item_features"]),
        "columns": 0 if feature_tables["item_features"].empty else len(feature_tables["item_features"].columns),
        "duplicate_rows": safe_duplicate_count(feature_tables["item_features"]),
    },
    {
        "table_name": "user_item_features",
        "rows": 0 if feature_tables["user_item_features"].empty else len(feature_tables["user_item_features"]),
        "columns": 0 if feature_tables["user_item_features"].empty else len(feature_tables["user_item_features"].columns),
        "duplicate_rows": safe_duplicate_count(feature_tables["user_item_features"]),
    },
    {
        "table_name": "item_cooccurrence",
        "rows": 0 if feature_tables["item_cooccurrence"].empty else len(feature_tables["item_cooccurrence"]),
        "columns": 0 if feature_tables["item_cooccurrence"].empty else len(feature_tables["item_cooccurrence"].columns),
        "duplicate_rows": safe_duplicate_count(feature_tables["item_cooccurrence"]),
    },
])

version_summary_df = pd.DataFrame([{
    "feature_store_version": FEATURE_VERSION,
    "created_ts": RUN_TS,
    "offline_store": rel_path(FEATURE_STORE_DB),
    "training_demo_rows": 0 if training_retrieval_df is None else len(training_retrieval_df),
    "inference_demo_rows": 0 if inference_retrieval_df is None else len(inference_retrieval_df),
}])

config_preview = read_json_preview(FEATURE_STORE_CONFIG_JSON, max_chars=10000)
metadata_json_preview = read_json_preview(FEATURE_METADATA_JSON, max_chars=10000)
schema_preview = read_text_preview(FEATURE_STORE_SCHEMA_SQL, max_lines=200, max_chars=12000)

validation_preview = read_text_preview(latest_validation_txt, max_lines=80, max_chars=10000)
validation_json_preview = read_json_preview(latest_validation_json, max_chars=10000) if latest_validation_json else "No validation JSON report found."
validation_log_preview = read_text_preview(latest_validation_log, max_lines=80, max_chars=10000) if latest_validation_log else "No validation event log found."
feature_store_log_preview = read_text_preview(FEATURE_STORE_LOG, max_lines=80, max_chars=10000)

feature_store_tree = build_tree_text(FEATURE_STORE_DIR, max_depth=5, max_items=200)
transformed_tree = build_tree_text(TRANSFORMED_ROOT if TRANSFORMED_ROOT.exists() else FEATURES_ROOT, max_depth=5, max_items=180)

asset_previews = []
for p in feature_store_assets[:5]:
    asset_previews.append({
        "name": p.name,
        "relative_path": rel_path(p),
        "preview": read_code_preview(p, max_lines=120, max_chars=9000),
    })

# ============================================================
# 9) TABLES
# ============================================================
team_data = [
    ["Team Member Name", "Team Member ID"],
    ["BANSHIDHAR RATH", "2025AE05346"],
    ["JITENDRA KUMAR TIWARI", "2025AE05518"],
    ["KATBA ANKIT CHIMANBHAI", "2025AE05229"],
    ["NAVEEN SURATHU", "2025AE05492"],
]

artifact_rows = [["Artifact", "Relative Path", "Last Modified", "Size"]]
for p in [
    events_file,
    prepared_interactions_file,
    products_file,
    latest_validation_txt,
    latest_validation_json,
    latest_validation_pdf,
    latest_fix_log,
    latest_reval_summary,
    latest_reval_issues,
    latest_validation_log,
    FEATURE_STORE_DB,
    FEATURE_STORE_CONFIG_JSON,
    FEATURE_METADATA_CSV,
    FEATURE_METADATA_JSON,
    FEATURE_STORE_SCHEMA_SQL,
    FEATURE_RETRIEVAL_TRAINING_CSV,
    FEATURE_RETRIEVAL_INFERENCE_CSV,
    FEATURE_STORE_LOG,
]:
    if p and Path(p).exists():
        info = file_info(Path(p))
        artifact_rows.append([
            info["name"],
            info["relative_path"],
            info["modified"],
            f"{info['size_kb']} KB",
        ])
if len(artifact_rows) == 1:
    artifact_rows.append(["No artifacts found", "-", "-", "-"])

asset_rows = [["Feature Store Asset", "Relative Path", "Last Modified", "Size"]]
for p in feature_store_assets[:20]:
    info = file_info(p)
    asset_rows.append([
        info["name"],
        info["relative_path"],
        info["modified"],
        f"{info['size_kb']} KB",
    ])
if len(asset_rows) == 1:
    asset_rows.append(["No feature store related asset found", "-", "-", "-"])

versioned_output_rows = [["Output", "Relative Path", "Type", "Last Modified", "Size"]]
for p in [
    FEATURE_STORE_DB,
    FEATURE_STORE_CONFIG_JSON,
    FEATURE_METADATA_CSV,
    FEATURE_METADATA_JSON,
    FEATURE_STORE_SCHEMA_SQL,
    FEATURE_RETRIEVAL_TRAINING_CSV,
    FEATURE_RETRIEVAL_INFERENCE_CSV,
]:
    if p and Path(p).exists():
        info = file_info(p)
        versioned_output_rows.append([
            info["name"],
            info["relative_path"],
            info["suffix"] or "N/A",
            info["modified"],
            f"{info['size_kb']} KB",
        ])
if len(versioned_output_rows) == 1:
    versioned_output_rows.append(["No generated feature store outputs found", "-", "-", "-", "-"])

team_table = make_wrapped_table(team_data, col_widths=[4.0 * inch, 2.0 * inch])

artifact_table = make_wrapped_table(
    artifact_rows,
    col_widths=[1.55 * inch, 3.35 * inch, 1.00 * inch, 0.60 * inch],
    path_cols=[1],
    file_cols=[0]
)

asset_table = make_wrapped_table(
    asset_rows,
    col_widths=[1.65 * inch, 3.25 * inch, 1.00 * inch, 0.60 * inch],
    path_cols=[1],
    file_cols=[0]
)

versioned_output_table = make_wrapped_table(
    versioned_output_rows,
    col_widths=[1.55 * inch, 2.95 * inch, 0.65 * inch, 1.05 * inch, 0.60 * inch],
    path_cols=[1],
    file_cols=[0]
)

feature_table_summary_table = df_to_wrapped_table(source_table_summary_df, max_rows=10)
version_summary_table = df_to_wrapped_table(version_summary_df, max_rows=10)

metadata_preview_df = feature_metadata_df.copy()
if not metadata_preview_df.empty:
    keep_cols = ["feature_view", "entity_keys", "feature_name", "dtype", "source_path", "version"]
    metadata_preview_df = metadata_preview_df[keep_cols]
feature_metadata_table = df_to_wrapped_table(metadata_preview_df, max_rows=30)

training_demo_table = df_to_wrapped_table(training_retrieval_df, max_rows=12)
inference_demo_table = df_to_wrapped_table(inference_retrieval_df, max_rows=12)

# ============================================================
# 10) BUILD PDF STORY
# ============================================================
story = []

story.append(Paragraph("07 Feature Store", title_style))
story.append(Paragraph("<b>Course Name:</b> Data Management for Machine Learning", meta_style))
story.append(Paragraph("<b>Assignment Title:</b> End-to-End Data Management Pipeline for a Recommendation System", meta_style))
story.append(Paragraph("<b>Assignment:</b> Group 51 - Data management for Machine Learning Group 51", meta_style))
story.append(Spacer(1, 10))

story.append(Paragraph("<b>Team Members</b>", heading_style))
story.append(team_table)
story.append(Spacer(1, 14))

story.append(Paragraph("1. Objective", heading_style))
story.append(Paragraph(
    "This report documents the feature store stage of the recommendation pipeline. It captures the feature store configuration, metadata registry, versioned storage artifacts, and sample feature retrieval for both training and inference.",
    body_style
))

story.append(Paragraph("2. Objective Coverage", heading_style))
story.append(Paragraph("• Implement a simple custom feature store using a metadata registry and SQLite offline store.", bullet_style))
story.append(Paragraph("• Document feature names, data sources, and transformation logic.", bullet_style))
story.append(Paragraph("• Support versioned retrieval for both training and inference workflows.", bullet_style))
story.append(Paragraph("• Capture project logs and evidence needed for a submission-ready PDF.", bullet_style))
story.append(Spacer(1, 10))

story.append(Paragraph("3. Supporting Project and Log Artifacts", heading_style))
story.append(Paragraph(
    "The following discovered files support this section, including source datasets, validation outputs, feature store artifacts, and retrieval demos. Long file paths are wrapped inside table cells using real line breaks only at safe separators.",
    body_style
))
story.append(artifact_table)
story.append(Spacer(1, 12))

story.append(Paragraph("4. Feature Store / Registry Assets in Project", heading_style))
story.append(asset_table)
story.append(Spacer(1, 12))

story.append(Paragraph("5. Feature Store Logic Summary", heading_style))
logic_notes = []
logic_notes.extend(source_notes if source_notes else [])
logic_notes.extend([
    "Feature store design uses a custom metadata registry instead of Feast.",
    "Offline store persists versioned feature tables in SQLite.",
    "Configuration is written to feature_store/registry/feature_store_config.json.",
    "Feature metadata is written to CSV and JSON registry files.",
    "Versioned retrieval uses *_v1 tables inside the feature store database.",
    "Training retrieval joins user, item, and user-item feature views.",
    "Inference retrieval demonstrates point-style feature lookup for a sample user and candidate items.",
])
for note in logic_notes:
    story.append(Paragraph(f"• {escape(note)}", bullet_style))
story.append(Spacer(1, 12))

story.append(Paragraph("6. Versioned Feature Store Outputs", heading_style))
story.append(versioned_output_table)
story.append(Spacer(1, 12))

story.append(Paragraph("7. Feature Table Summary", heading_style))
story.append(feature_table_summary_table)
story.append(Spacer(1, 12))

story.append(Paragraph("8. Feature Store Version Summary", heading_style))
story.append(version_summary_table)
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("9. Feature Metadata Documentation", heading_style))
story.append(Paragraph(
    "The metadata registry below documents feature view, entity keys, feature name, data type, source path, and version.",
    body_style
))
story.append(feature_metadata_table)
story.append(Spacer(1, 12))

story.append(Paragraph("10. Sample Training Feature Retrieval", heading_style))
story.append(training_demo_table)
story.append(Spacer(1, 12))

story.append(Paragraph("11. Sample Inference Feature Retrieval", heading_style))
story.append(inference_demo_table)
story.append(Spacer(1, 12))

story.append(Paragraph("12. Feature Store Folder Structure", heading_style))
story.append(Paragraph("<b>feature_store/</b>", meta_style))
story.append(Preformatted(wrap_block_text(feature_store_tree, width=92), code_style))
story.append(Spacer(1, 8))
story.append(Paragraph("<b>data/transformed or data/features</b>", meta_style))
story.append(Preformatted(wrap_block_text(transformed_tree, width=92), code_style))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("13. Feature Store Configuration Preview", heading_style))
story.append(Preformatted(wrap_block_text(config_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(Paragraph("14. Feature Metadata JSON Preview", heading_style))
story.append(Preformatted(wrap_block_text(metadata_json_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(Paragraph("15. Feature Store SQL Schema", heading_style))
story.append(Preformatted(wrap_block_text(schema_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("16. Feature Store Code / Asset Preview", heading_style))
if asset_previews:
    for item in asset_previews:
        story.append(Paragraph(f"Asset: {escape(item['name'])}", sub_heading_style))
        story.append(Paragraph(f"<b>Path:</b> {escape(item['relative_path'])}", meta_style))
        story.append(Preformatted(wrap_block_text(item["preview"], width=95), code_style))
        story.append(Spacer(1, 10))
else:
    story.append(Paragraph("No feature store related notebook, script, SQL, or registry preview is available.", body_style))
story.append(Spacer(1, 8))

story.append(Paragraph("17. Validation / Pipeline Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("18. Data Quality Report Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_json_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("19. Validation Event Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_log_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("20. Feature Store Event Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(feature_store_log_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("21. Conclusion", heading_style))
story.append(Paragraph(
    "This PDF consolidates the feature store deliverables for the assignment, including a simple versioned feature store implementation, feature metadata documentation, and sample training and inference retrieval demonstrations.",
    body_style
))

# ============================================================
# 11) BUILD PDF
# ============================================================
def build_pdf(path):
    doc = SimpleDocTemplate(
        str(path),
        pagesize=A4,
        rightMargin=0.50 * inch,
        leftMargin=0.50 * inch,
        topMargin=0.55 * inch,
        bottomMargin=0.55 * inch,
    )
    doc.build(story)

try:
    build_pdf(OUTPUT_PATH)
    print(f"\nPDF created successfully: {OUTPUT_PATH}")
except PermissionError:
    alt_path = PROJECT_ROOT / f"07 Feature Store- DM4ML-Group51-{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    build_pdf(alt_path)
    print("\nOriginal PDF is likely open or locked.")
    print(f"Saved alternate file instead: {alt_path}")


PROJECT_ROOT: C:\Users\barath\recomart-pipeline
FEATURE_STORE_DIR: C:\Users\barath\recomart-pipeline\feature_store
FEATURE_STORE_DB: C:\Users\barath\recomart-pipeline\feature_store\offline_store\feature_store.db
OUTPUT_PATH: C:\Users\barath\recomart-pipeline\07 Feature Store- DM4ML-Group51.pdf


C:\Users\barath\AppData\Local\Temp\ipykernel_27184\1207367726.py:390: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, nrows=nrows)



PDF created successfully: C:\Users\barath\recomart-pipeline\07 Feature Store- DM4ML-Group51.pdf
